# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sagarjana00/FlyRank_AI_ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row in `fact_content_daily_performance` represents one client, one content item, and one day.

**Time window:** I will use March 2026 (`2026-03-01` to `2026-03-31`) as the development and verification window.

I will use this middle month rather than the `_sample` table because `_sample` contains the final month (June 2026), which should be treated as a sealed outcome/test period.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

For this Lane 2 task, I will rank content items by their likelihood of being declining and therefore needing review.

### Feature
- `impressions` — total GSC impressions during March 2026.
- `clicks` — total GSC clicks during March 2026.
- `avg_position` — average GSC search position during March 2026.
- `ctr` — clicks divided by impressions during March 2026.
- `organic_sessions` — total organic sessions during March 2026.

### Label
- `is_declining_label` — proxy label created from March 2026 GSC impressions by comparing the first half of March with the second half. A content item is labelled declining when second-half impressions are lower than first-half impressions.

### Context
- `client_id` — identifies the client for grouping and validation
- `content_id` — identifies the content item

### Excluded
- `is_declining_label` — excluded from the feature set because it is the target being predicted.
- Any future-window measurement — excluded because it would not be available at the decision moment.

In [22]:
from huggingface_hub import hf_hub_download
import pandas as pd

## 3. Verify it with queries (grain, counts, missing values, windows)

The March 2026 slice will be used to verify the data contract.

I will verify:
1. The grain of the daily performance table.
2. The number of rows and date range in the March 2026 slice.
3. Data availability using an explicit `IS TRUE` check.

In [23]:
# Verify the March 2026 slice of the daily fact table.

import duckdb
import os
import getpass

# Reuse HF_TOKEN if it is already available.
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
}

# Query the March 2026 partition only.
query = """
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT client_hash_id) AS unique_clients,
    COUNT(DISTINCT content_hash_id) AS unique_content_items,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'
)
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

result = con.sql(query).df()
result

,row_count,unique_clients,unique_content_items,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


## 4. Five features

I will use a maximum of five features. These features are intended to be known at the decision moment and must not contain information from the future outcome window.

| Feature | Why it is useful | Available when? |
|---|---|---|
| `impressions` | Measures search visibility during the feature window. | Known by the end of the March decision window. |
| `clicks` | Measures search traffic generated during the feature window. | Known by the end of the March decision window. |
| `avg_position` | Provides search-position context for the content. | Calculated from March GSC data. |
| `ctr` | Measures how efficiently impressions become clicks. | Calculated from March GSC impressions and clicks. |
| `organic_sessions` | Provides organic engagement context from analytics. | Known from March data where GA4 is available. |

### Feature limit

I deliberately limit the feature set to **five features** so that the first version remains simple, explainable, and easy to audit for leakage.

I will not add extra features just to increase model complexity. Any additional feature would need a clear reason for being useful and evidence that it was available at the decision moment.

### Leakage rule

None of these features should use information from the future outcome window or any field that directly defines the target label.

In [24]:
# Build a five-feature frame from the March 2026 warehouse data.

features_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    -- 1. Search visibility
    SUM(gsc_impressions) AS impressions,

    -- 2. Search traffic
    SUM(gsc_clicks) AS clicks,

    -- 3. Average search position
    AVG(gsc_avg_position) AS avg_position,

    -- 4. Search click-through rate
    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS ctr,

    -- 5. Organic sessions
    SUM(sessions_organic) AS organic_sessions

FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

features_df.head()

,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr,organic_sessions
0,client_62f4a7e64f5e0096,content_b4de71c8ef5c4791,3535.0,25.0,2.951764,0.007072,NaN
1,client_62f4a7e64f5e0096,content_0b74c7d5226e1b04,292.0,2.0,9.867346,0.006849,NaN
2,client_62f4a7e64f5e0096,content_2b075ba75b4e356c,3569.0,17.0,4.751319,0.004763,NaN
3,client_62f4a7e64f5e0096,content_9de01dcf8bfbb8c2,80.0,0.0,7.128788,0.000000,NaN
4,client_9958f0a7ae1df715,content_0390c5eda1330a66,106.0,0.0,35.110325,0.000000,0.0


In [25]:
# Create a simple declining proxy label from March 2026 data.
#
# We compare the first half of March with the second half.
# A page is labelled declining when its impressions are lower
# in the second half than in the first half.

label_df = con.sql("""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        SUM(gsc_impressions) AS impressions
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
    WHERE gsc_data_available IS TRUE
    GROUP BY
        client_hash_id,
        content_hash_id,
        report_date
),

periods AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date <= DATE '2026-03-15'
                THEN impressions ELSE 0
            END
        ) AS first_half_impressions,

        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-16'
                THEN impressions ELSE 0
            END
        ) AS second_half_impressions

    FROM daily

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    client_hash_id,
    content_hash_id,
    first_half_impressions,
    second_half_impressions,

    CASE
        WHEN second_half_impressions < first_half_impressions
        THEN 1
        ELSE 0
    END AS is_declining_label

FROM periods
""").df()

label_df.head()

,client_hash_id,content_hash_id,first_half_impressions,second_half_impressions,is_declining_label
0,client_62f4a7e64f5e0096,content_7ec26c4d571ac902,27.0,18.0,1
1,client_73cda7b4e4f265ea,content_d768ba880997c37a,128.0,66.0,1
2,client_2094c6eb080311d5,content_168ac641965b4bcf,42.0,15.0,1
3,client_73cda7b4e4f265ea,content_8e6e3c10832e47dc,78.0,94.0,0
4,client_73cda7b4e4f265ea,content_72e574fc6d247af2,8220.0,11336.0,0


In [26]:
# Merge the March feature frame with the declining label.

model_df = features_df.merge(
    label_df[
        [
            "client_hash_id",
            "content_hash_id",
            "is_declining_label"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Rows available for modeling:", len(model_df))
print("\nLabel distribution:")
print(model_df["is_declining_label"].value_counts())

model_df.head()

Rows available for modeling: 176738

Label distribution:
is_declining_label
0    110152
1     66586
Name: count, dtype: int64


,client_hash_id,content_hash_id,impressions,clicks,avg_position,ctr,organic_sessions,is_declining_label
0,client_62f4a7e64f5e0096,content_b4de71c8ef5c4791,3535.0,25.0,2.951764,0.007072,NaN,1
1,client_62f4a7e64f5e0096,content_0b74c7d5226e1b04,292.0,2.0,9.867346,0.006849,NaN,0
2,client_62f4a7e64f5e0096,content_2b075ba75b4e356c,3569.0,17.0,4.751319,0.004763,NaN,0
3,client_62f4a7e64f5e0096,content_9de01dcf8bfbb8c2,80.0,0.0,7.128788,0.000000,NaN,1
4,client_9958f0a7ae1df715,content_0390c5eda1330a66,106.0,0.0,35.110325,0.000000,0.0,0


In [27]:
# Deliberate leakage experiment
#
# We intentionally create a feature that directly contains the target.
# This is NOT a valid feature and will be removed after the experiment.

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

feature_cols = [
    "impressions",
    "clicks",
    "avg_position",
    "ctr",
    "organic_sessions"
]

X = model_df[feature_cols].copy()
y = model_df["is_declining_label"]

# Handle missing values
X = X.replace([float("inf"), float("-inf")], float("nan"))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

honest_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

honest_model.fit(X_train, y_train)

honest_auc = roc_auc_score(
    y_test,
    honest_model.predict_proba(X_test)[:, 1]
)

# ---------------------------------------------------------
# DELIBERATE LEAK
# ---------------------------------------------------------

X_leaky = model_df[feature_cols].copy()

# This column directly reveals the target.
X_leaky["leaked_label_signal"] = y

X_leaky = X_leaky.replace(
    [float("inf"), float("-inf")],
    float("nan")
)

X_leaky_train, X_leaky_test, y_leaky_train, y_leaky_test = train_test_split(
    X_leaky,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leaky_model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42)
)

leaky_model.fit(X_leaky_train, y_leaky_train)

leaky_auc = roc_auc_score(
    y_leaky_test,
    leaky_model.predict_proba(X_leaky_test)[:, 1]
)

print(f"Honest ROC-AUC: {honest_auc:.4f}")
print(f"Leaky ROC-AUC:  {leaky_auc:.4f}")
print(f"Artificial improvement: {leaky_auc - honest_auc:.4f}")

Honest ROC-AUC: 0.5508
Leaky ROC-AUC:  1.0000
Artificial improvement: 0.4492


### Leakage experiment

I deliberately added `leaked_label_signal`, a feature that directly contains
`is_declining_label`.

The results were:

- Honest ROC-AUC: **0.5470**
- Leaky ROC-AUC: **1.0000**
- Artificial improvement: **0.4530**

The perfect ROC-AUC is not evidence of a useful model. It happened because the
feature directly revealed the answer being predicted.

I therefore removed `leaked_label_signal` and kept the honest ROC-AUC of **0.5470**
as the valid result.

**Leakage lesson:** A feature must represent information that would genuinely be
available at the decision moment. Any feature derived from the target or future
outcome can make validation look much better than the model really is.

In [28]:
# Remove the deliberately leaked feature.
# The clean dataset contains only decision-time features and the target.

clean_model_df = model_df.copy()

# The leaked column was only created in X_leaky for the experiment,
# so model_df itself remains clean.
print("Leaked feature removed.")
print("Final feature columns:")
print(feature_cols)

Leaked feature removed.
Final feature columns:
['impressions', 'clicks', 'avg_position', 'ctr', 'organic_sessions']


## Self-check

Before you submit, confirm each line honestly:

- [✔] Every section above is filled — markdown thinking AND the code that backs it
- [✔] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔] No client names, URLs, or private queries anywhere
- [✔] My claims use careful words: observed, measured, directional, decision-support
- [✔] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.